[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/08_knn/03_scaling_and_choosing_k.ipynb)

# Bài 03 — Vì sao KNN gần như bắt buộc phải scaling? Chọn k thế nào?

**🎯 Mục tiêu**
- Trả lời câu hỏi kinh điển: **vì sao KNN gần như bắt buộc phải scaling?** — bằng thực nghiệm.
- Chọn $k$ bằng cross-validation (validation curve); các quy tắc kinh nghiệm.
- Xử lý feature phân loại trong KNN; Pipeline hoàn chỉnh.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/08_knn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, train_test_split, validation_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

from ml_course.visualization import plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Vì sao scaling gần như bắt buộc?

KNN dựa hoàn toàn vào **khoảng cách**, mà khoảng cách cộng gộp chênh lệch của **mọi** feature:

$$d(\mathbf a, \mathbf b) = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2 + \dots}$$

Nếu thu nhập (đơn vị triệu, chênh lệch hàng chục) và tuổi (chênh lệch vài năm) cùng có mặt, khoảng cách gần như **chỉ do thu nhập** quyết định — tuổi bị bỏ qua dù có thể rất quan trọng. **Đổi đơn vị** (triệu → nghìn) thậm chí làm thay đổi hoàn toàn láng giềng!

Thí nghiệm: nhãn phụ thuộc **chủ yếu vào tuổi**, nhưng thu nhập có thang đo lớn gấp hàng trăm lần:

In [ ]:
n = 600
age = rng.uniform(18, 70, n)                          # năm
income = rng.lognormal(np.log(15_000), 0.5, n)        # nghìn VND / tháng — thang đo rất lớn
y = (age + rng.normal(0, 5, n) > 42).astype(int)      # nhãn gần như chỉ do tuổi quyết định
X = np.c_[age, income]

for name, model in [("không scale", KNeighborsClassifier(15)),
                    ("StandardScaler", make_pipeline(StandardScaler(), KNeighborsClassifier(15))),
                    ("MinMaxScaler", make_pipeline(MinMaxScaler(), KNeighborsClassifier(15))),
                    ("RobustScaler", make_pipeline(RobustScaler(), KNeighborsClassifier(15)))]:
    print(f"{name:15s} accuracy (5-fold CV) = {cross_val_score(model, X, y, cv=5).mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
raw = KNeighborsClassifier(15).fit(X, y)
plot_decision_boundary(raw, X, y, ax=axes[0], padding=1)
axes[0].set_title("Không scale: ranh giới theo THU NHẬP (sai)")
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(15)).fit(X, y)
plot_decision_boundary(scaled, X, y, ax=axes[1], padding=1)
axes[1].set_title("Đã scale: ranh giới theo TUỔI (đúng)")
for ax in axes:
    ax.set_xlabel("tuổi")
    ax.set_ylabel("thu nhập")
plt.show()

Không scale, model gần như đoán mò (≈ tỉ lệ lớp); sau khi scale, accuracy tăng vọt. Đây là lý do câu trả lời là **"gần như bắt buộc"**:
- Bắt buộc khi các feature có đơn vị/thang đo khác nhau (hầu hết dữ liệu thật).
- Ngoại lệ: mọi feature đã cùng thang đo và cùng mức độ quan trọng (vd: pixel ảnh 0–255, tần suất từ đã chuẩn hoá).

> ⚠️ Scaling cũng ngầm định **mọi feature quan trọng như nhau**. Nếu có feature vô ích, nó vẫn "đóng góp" vào khoảng cách → cần feature selection (bài 04, chương 18) hoặc học metric.

## 2. Chọn k bằng cross-validation

**Validation curve:** accuracy train và CV theo $k$.

In [ ]:
from ml_course.datasets import make_classification_2d

Xm, ym = make_classification_2d("moons", n=600, noise=0.35, seed=4)
ks = np.arange(1, 101, 2)
train_scores, val_scores = validation_curve(KNeighborsClassifier(), Xm, ym, param_name="n_neighbors",
                                            param_range=ks, cv=5)
best_k = ks[np.argmax(val_scores.mean(1))]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, train_scores.mean(1), label="train")
ax.plot(ks, val_scores.mean(1), label="validation (5-fold CV)")
ax.axvline(best_k, color="gray", ls="--", label=f"k tốt nhất = {best_k}")
ax.set_xlabel("k")
ax.set_ylabel("accuracy")
ax.legend()
ax.set_title("k nhỏ: overfit (train ≫ validation) — k lớn: underfit")
plt.show()

**Quy tắc kinh nghiệm** (điểm khởi đầu, sau đó dùng CV):
- $k \approx \sqrt{n}$;
- bài toán 2 lớp: chọn $k$ **lẻ** để tránh hoà;
- dữ liệu nhiều nhiễu → $k$ lớn hơn; ranh giới phức tạp, dữ liệu nhiều → $k$ nhỏ hơn;
- `weights="distance"` thường làm kết quả ít nhạy với $k$.

## 3. Tìm đồng thời k, weights và metric

In [ ]:
from sklearn.model_selection import GridSearchCV

from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=3000, seed=1)
num = ["tenure_months", "monthly_charges", "support_calls", "n_products"]
Xc, yc = churn[num].to_numpy(float), churn["churn"].to_numpy()
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = GridSearchCV(pipe, {"kneighborsclassifier__n_neighbors": [5, 15, 31, 61, 101],
                           "kneighborsclassifier__weights": ["uniform", "distance"],
                           "kneighborsclassifier__p": [1, 2]}, cv=5, scoring="roc_auc").fit(Xc, yc)
print("Tốt nhất:", grid.best_params_, f"| ROC-AUC = {grid.best_score_:.4f}")
pd.DataFrame(grid.cv_results_)[["param_kneighborsclassifier__n_neighbors", "param_kneighborsclassifier__weights",
                                "param_kneighborsclassifier__p", "mean_test_score"]].sort_values("mean_test_score", ascending=False).head(6).round(4)

## 4. Feature phân loại trong KNN

- **One-hot** rồi scale: mỗi category khác nhau đóng góp khoảng cách $\sqrt 2$ (hai cột khác nhau) — cần cân nhắc trọng số so với feature số.
- **Hamming / Gower distance** cho dữ liệu hỗn hợp.
- Feature thứ bậc → ordinal encoding rồi scale.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat = ["contract", "internet_service", "payment_method"]
pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
for name, cols, prep in [("chỉ feature số", num, StandardScaler()), ("số + one-hot", num + cat, pre)]:
    model = make_pipeline(prep, KNeighborsClassifier(61))
    auc = cross_val_score(model, churn[cols], yc, cv=5, scoring="roc_auc").mean()
    print(f"{name:15s} ROC-AUC = {auc:.4f}")

## 🧠 Kiểm tra nhanh

1. Giải thích bằng một câu: vì sao KNN gần như bắt buộc phải scaling?
2. Đổi đơn vị thu nhập từ "triệu" sang "nghìn" có làm thay đổi dự đoán của KNN không scale? Còn Decision Tree?
3. Validation curve: train accuracy = 1.0 tại $k = 1$. Đây có phải tín hiệu tốt?
4. Bài toán 2 lớp, $n = 900$ mẫu. Chọn $k$ khởi đầu bao nhiêu?

## 📌 Tóm tắt

- KNN dùng khoảng cách → feature có thang đo lớn chi phối → **scale trong Pipeline**.
- Chọn $k$ (và weights, metric) bằng CV; $k \approx \sqrt n$, $k$ lẻ cho 2 lớp là điểm khởi đầu.
- Feature phân loại: one-hot + scale, hoặc metric chuyên dụng.